In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import numpy as np
import os
from tqdm import tqdm
import random
import torchio as tio
from models.unet_3d import UNet3D
from utils.unets_helper_functions import (
    PatchDataset,
    dice_loss
)


c:\Users\dhanu\Desktop\mini-project-1\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()


In [4]:
train_dataset = PatchDataset(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 6,
                        patch_size = 80,
                        augment=False)

val_dataset = PatchDataset(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = 80,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=2,    
    pin_memory=True
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet3D().to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([0.1, 1, 1, 1, 1, 1, 1]).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights)


scaler = torch.cuda.amp.GradScaler()

C:\Users\dhanu\AppData\Local\Temp\ipykernel_18392\3241659940.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060 Laptop GPU


In [ ]:
EPOCHS = 40
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "unet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # train
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.long().to(device)


        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = dice_loss(outputs, labels) + ce_loss(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    #  Validation 
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(val_loader):
            images = images.to(device)
            labels = labels.long().to(device)

            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = dice_loss(outputs, labels) + ce_loss(outputs, labels)

            val_loss += loss.item()

            #  Debug predicted classes (only first batch)
            if batch_idx == 0:
                preds = torch.argmax(outputs, dim=1)
                print("Unique predicted classes:", torch.unique(preds))

    val_loss /= len(val_loader)


    print(f"Epoch {epoch}: Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f}")

    #  Save best model 
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))
        
        print(" Saved Best Model")

    #  Save checkpoints after 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss
        }, os.path.join(SAVE_DIR, f"unet_checkpoint_epoch_{epoch+1}.pth"))

        print(" Checkpoint Saved")


  0%|          | 0/99 [00:00<?, ?it/s]C:\Users\dhanu\AppData\Local\Temp\ipykernel_18392\1447452787.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 99/99 [03:10<00:00,  1.93s/it]
C:\Users\dhanu\AppData\Local\Temp\ipykernel_18392\1447452787.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Unique predicted classes: tensor([0, 1, 2, 3, 4, 5, 6], device='cuda:0')
Epoch 0: Train Loss 2.6074 | Val Loss 2.3477
 Saved Best Model


100%|██████████| 99/99 [03:16<00:00,  1.99s/it]


Unique predicted classes: tensor([0, 1, 2, 3, 4, 5, 6], device='cuda:0')
Epoch 1: Train Loss 2.4000 | Val Loss 2.2231
 Saved Best Model


100%|██████████| 99/99 [03:55<00:00,  2.38s/it]


...

Unique predicted classes: tensor([0, 1, 2, 5, 6], device='cuda:0')
Epoch 37: Train Loss 0.9983 | Val Loss 1.0608


100%|██████████| 99/99 [03:43<00:00,  2.25s/it]


Unique predicted classes: tensor([0, 1, 2, 4], device='cuda:0')
Epoch 38: Train Loss 1.0012 | Val Loss 0.9429
 Saved Best Model


100%|██████████| 99/99 [03:06<00:00,  1.89s/it]


Unique predicted classes: tensor([0, 1, 5, 6], device='cuda:0')
Epoch 39: Train Loss 0.9783 | Val Loss 0.9308
 Saved Best Model
 Checkpoint Saved


In [8]:
model_path = "../experiments/unet_fold0/best_model.pth"

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded for testing.")

Model loaded for testing.


C:\Users\dhanu\AppData\Local\Temp\ipykernel_18392\3274664165.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)


In [9]:
from utils.unets_helper_functions import evaluate_full_volume

mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.7330567082155495), np.float64(0.6514740111991136), np.float64(3.752345201679755e-09), np.float64(0.4741581223785155), np.float64(0.7413092553125876), np.float64(0.8184212706339069)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.7052927295864045), np.float64(0.6036754219990975), np.float64(3.4340659222731253e-09), np.float64(0.5825828148959905), np.float64(0.7907593678618551), np.float64(0.8342906024938453)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.6279803647379075), np.float64(0.7736522448352312), np.float64(3.409478338187936e-09), np.float64(0.6067372475424748), np.float64(0.7813971004949494), np.float64(0.8078358209222772)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.696346866222106), np.float64(0.5753273607337509), np.float64(4.496402

Train Loss 0.9783 | Val Loss 0.9308

| Organ       | Dice | Interpretation |
| ----------- | ---- | -------------- |
| Mandible    | 0.70 | Good           |
| Brainstem   | 0.68 | Good           |
| SpinalCord  | 0.00 | Missing        |
| Parotid_L   | 0.56 | Acceptable     |
| Parotid_R   | 0.74 | Good           |
| Oral Cavity | 0.75 | Good           |


SpinalCord Fails maybe beacuse It is extremely thin structure
Very small voxel count
Hard to capture with 80³ patches
May not be sufficiently sampled

our foreground sampling in patch dataset:
`fg_indices = np.argwhere(label > 0)`
This samples ALL organs equally.
But spinal cord voxels are extremely fewer.
So probability of sampling spinal cord center is very low.
So making changes in patch dataset class from attention unet